# Baseline 04b — RandomForest sobre el subset de features

Variante mínima del baseline tabular sobre el subset Italia (85 951 parcelas, 17 índices espectrales × 9 estadísticos + FFT NDVI + 8 atributos fenológicos). Sirve como piloto del patrón `setup_notebook` + `train_baseline_three_models` que reúsan los demás cuadernos.

**Pregunta**: ¿qué F1-macro out-of-fold consigue RandomForest puro sobre este subset, con validación cruzada espacial de 5 particiones y un buffer anti-fuga de 1 km entre ellas?

## Requisitos

- `data/test_fixtures/feature_selection_parcels_subset.parquet` presente (descargable vía `dvc pull`).
- `data/processed/pastis_parcels_full.geoparquet` presente (generado por el pipeline de muestreo de parcelas).


In [ ]:
FEATURES_PATH = "data/test_fixtures/feature_selection_parcels_subset.parquet"
PARCELS_GEOPARQUET = "data/processed/pastis_parcels_full.geoparquet"
FIGURES_SUBDIR = "us-023-preview/04b_baseline"
REPORTS_SUBDIR = "baseline/04b_baseline"
K_FOLDS = 5
BUFFER_KM = 1.0
RANDOM_STATE = 42


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Bootstrap: localizar el repo root buscando pyproject.toml
_HERE = Path.cwd().resolve()
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "pyproject.toml").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from ml.utils.notebook_bootstrap import setup_notebook
from IPython.display import Markdown, display

env = setup_notebook(
    figures_subdir=FIGURES_SUBDIR,
    reports_subdir=REPORTS_SUBDIR,
)
display(Markdown(env.summary_markdown()))


## Carga del dataset

Unimos el subset de características con la metadata real (clase, `patch_id`, fold, área) desde el geoparquet de parcelas. El resultado tiene `parcel_id` como `pl.Utf8`, esquema canónico del proyecto.

In [ ]:
import polars as pl
from ml.utils.baseline_notebook_helpers import (
    load_features_dataset_with_meta,
    train_baseline_three_models,
    build_model_comparison_table,
)
from ml.utils.class_distribution import (
    class_distribution_report,
    recommend_threshold,
)

df = load_features_dataset_with_meta(
    path=FEATURES_PATH,
    parcels_geoparquet=PARCELS_GEOPARQUET,
)
parcel_id_dtype = df.schema['parcel_id']
display(Markdown(
    f"**Dataset cargado**: `{df.height:,}` parcelas x "
    f"`{df.width}` columnas. `parcel_id` dtype: `{parcel_id_dtype}`."
))
display(df.head(5))


## Distribución de clases

Reportamos las 18 clases con su conteo, proporción y banda de soporte (`high` / `med` / `low` / `very_low`). El umbral se deriva del percentil 25 de la distribución, no de un valor fijo, para evitar marcar como minoritarias a clases que sí tienen soporte suficiente.

In [ ]:
report = class_distribution_report(df)
display(report)

threshold_p25 = recommend_threshold(report, method='p25')
threshold_p50 = recommend_threshold(report, method='p50')
display(Markdown(
    f"**Umbral sugerido**: percentil 25 = `{threshold_p25}`, "
    f"percentil 50 = `{threshold_p50}` parcelas. "
    "Las clases por debajo del umbral tienen soporte débil "
    "y se resaltan en color en los gráficos."
))


## Entrenamiento de RandomForest, XGBoost y LightGBM

Cada modelo se entrena con la misma validación cruzada espacial: 5 particiones determinadas por bloques H3 + KMeans y un buffer de 1 km que separa train y test para evitar fuga espacial. `train_baseline_three_models` devuelve métricas out-of-fold, tiempo de entrenamiento y la tabla comparativa.

In [ ]:
rows = train_baseline_three_models(
    df,
    models=('rf', 'xgb', 'lgbm'),
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    random_state=RANDOM_STATE,
)
comparison_path = env.reports_dir / 'model_comparison_04b.parquet'
comparison = build_model_comparison_table(rows, output_path=comparison_path)
display(Markdown(f'**Tabla comparativa guardada**: `{comparison_path}`'))
display(comparison)


## Comparativa de modelos y soporte por clase

In [ ]:
import matplotlib.pyplot as plt
from ml.eval.reencuadre_plots import (
    plot_class_support_bars,
    plot_model_comparison_bars,
)

metric_by_model = {r.model: r.f1_macro for r in rows}
fig1 = plot_model_comparison_bars(
    metric_by_model,
    baseline_value=0.40,
    baseline_label='referencia previa (F1-macro 0.40)',
    title='F1-macro out-of-fold: RandomForest, XGBoost, LightGBM',
)
fig1.savefig(env.figures_dir / 'model_comparison_04b.png', bbox_inches='tight')
display(fig1)
plt.close(fig1)

fig2 = plot_class_support_bars(
    report.rename({'n_parcels': 'len'}),
    weak_threshold=threshold_p25,
    title=f'Soporte por clase (umbral P25 = {threshold_p25} parcelas)',
)
fig2.savefig(env.figures_dir / 'class_support_04b.png', bbox_inches='tight')
display(fig2)
plt.close(fig2)


## F1 por clase del mejor modelo

In [ ]:
from ml.eval.reencuadre_plots import plot_per_class_f1
from ml.train.baseline import train_one_model
from ml.ingest.pastis_loader import PASTIS_R_CLASSES
from ml.train.baseline import evaluate_with_spatial_cv, build_estimator

best_model = comparison['model'][0]
display(Markdown(f'**Mejor modelo**: `{best_model}` (F1-macro `{comparison["f1_macro"][0]:.4f}`)'))

# Reentrenamos brevemente el mejor modelo para conseguir y_pred_oof.
best_result = train_one_model(
    df,
    model=best_model,
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    random_state=RANDOM_STATE,
)
# Recuperamos las predicciones out-of-fold via evaluate_with_spatial_cv
_cv_metrics, y_true_oof, y_pred_oof = evaluate_with_spatial_cv(
    df,
    lambda: build_estimator(best_model, best_result.best_params),
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    random_state=RANDOM_STATE,
)

# Decodificamos las etiquetas para nombres legibles
class_names = {int(c): PASTIS_R_CLASSES.get(int(c), f'class_{int(c)}') for c in best_result.label_classes}
fig3 = plot_per_class_f1(
    y_true_oof,
    y_pred_oof,
    class_labels=list(range(len(best_result.label_classes))),
    class_names={i: class_names[c] for i, c in enumerate(best_result.label_classes)},
    weak_threshold=0.10,
    title=f'F1 por clase ({best_model}) out-of-fold',
)
fig3.savefig(env.figures_dir / 'per_class_f1_04b.png', bbox_inches='tight')
display(fig3)
plt.close(fig3)


## Conclusiones

Esta libreta valida el patrón de arranque (`setup_notebook` + `baseline_notebook_helpers`) y produce una primera referencia comparativa de los tres modelos sobre el subset de características.

- **Tabla `model_comparison_04b.parquet`**: F1-macro, F1-weighted, mIoU, accuracy, kappa y tiempo de entrenamiento por modelo. Sirve como referencia local para detectar regresiones al incorporar bloques opcionales.
- **Umbral de soporte por percentil 25**: las clases minoritarias quedan resaltadas sin marcar artificialmente a todas como débiles.
- **F1 por clase del mejor modelo**: identifica qué clases concentran el error y sugiere si conviene agrupar por ciclo fenológico mediante `merge_to_phenological_groups`.

## Lo que sigue

- `04_baseline.ipynb` aplica el mismo patrón sobre el conjunto completo de características (AlphaEarth + ERA5 + SRTM + índices).
- `05_reencuadre_fenologico.ipynb` cuantifica el aporte de los bloques opcionales (FarSLIP, descripción fenológica textual, firma espectral REP).
- `Avance3.Equipo17.ipynb` selecciona y nombra el conjunto de características ganador con `select_winning_features`.